# Remap arbitrary grids to HEALPix

This notebook reads a NetCDF dataset lazily with `xarray`, selects only the requested variables, timesteps, and vertical levels, and remaps every selected field to the centers of a user-selected HEALPix grid. It accepts rectilinear latitude–longitude grids, curvilinear grids, and unstructured grids such as ICON when longitude and latitude coordinates are present in the file.

Rectilinear grids use vectorized linear interpolation from `xarray`. Curvilinear and unstructured grids use a seam-aware piecewise-linear triangulation, followed by spherical nearest-neighbor filling only for targets outside the triangulation or next to missing source values. Time and level slices are interpolated independently. The resulting dataset retains global, variable, time, and level metadata and is written to a consolidated Zarr store with a `cell` dimension and HEALPix center coordinates.

## 1. Environment and imports

Run this notebook from the `climgen` environment (`module load python3`, then `source activate climgen`). This cell configures writable plotting caches and imports only packages available in that environment. A missing `healpy`, SciPy, NetCDF, or Zarr dependency will fail here before any data is processed.

In [ ]:
import os
import platform
import re
from datetime import datetime, timezone
from pathlib import Path

# Keep cache writes off potentially read-only home filesystems.
CACHE_ROOT = Path('/tmp') / f'remap_any_to_healpix_cache_{os.getuid()}'
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(CACHE_ROOT / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_ROOT))

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import xarray as xr
from scipy.interpolate import LinearNDInterpolator
from scipy.spatial import Delaunay, cKDTree
from tqdm.auto import tqdm

display(pd.Series({
    'Python': platform.python_version(),
    'xarray': xr.__version__,
    'SciPy': scipy.__version__,
    'healpy': hp.__version__,
}).to_frame('version'))

## 2. Editable remapping configuration

This is the only cell most users need to edit. The `VARIABLES` mapping follows the other compression notebooks: `timesteps` accepts non-negative integers and end-exclusive ranges such as `"5-100"`; variables can be organized under `2D` and `3D`; and `level_indices` accepts one integer, a list of integers, or `None`. Omitting `timesteps` uses every timestep. Omitting all variable groups uses every numeric spatial data variable. Omitting `level_indices` uses every level.

Coordinate names and the vertical dimension are normally detected from CF metadata and common names. Use the optional overrides only for unconventional files. `HEALPIX_NESTED=True` matches the ordering used by the compression notebooks. `OVERWRITE_OUTPUT=False` protects an existing output store.

In [ ]:
INPUT_PATH = Path(
    '/work/bd1560/k204233/FieldSpaceNN/data/netcdf/mpi-esm-lr/eva-ens/2mt/'
    'eva0ssi101_echam6_BOT_mm_2mt_1991-1995.nc'
)
HEALPIX_LEVEL = 5
OUTPUT_PATH = Path(os.environ.get(
    'HEALPIX_REMAP_OUTPUT',
    str(INPUT_PATH.parent / f'{INPUT_PATH.stem}_hpx{HEALPIX_LEVEL}.zarr'),
))

VARIABLES = {
    'timesteps': ['0-10'],  # First ten timesteps; delete this entry to use all.
    # With no 2D/3D entries, every numeric spatial variable and every level is used.
    # '2D': {'tas': {}},
    # '3D': {'ua': {'level_indices': [0, 2, 5]}},
}

LATITUDE_NAME = None       # Examples: 'lat', 'latitude', or 'clat'
LONGITUDE_NAME = None      # Examples: 'lon', 'longitude', or 'clon'
TIME_DIMENSION = None      # Example: 'time'
LEVEL_DIMENSIONS = {}      # Optional mapping such as {'ua': 'lev'}
HEALPIX_NESTED = True
OUTPUT_DTYPE = 'float32'
OVERWRITE_OUTPUT = False

## 3. Selection and coordinate helpers

These helpers expand timestep ranges, normalize per-variable level selections, and discover longitude, latitude, and time coordinates from CF attributes or conventional names. Explicit selections are validated before interpolation. For files with several possible horizontal grids, set `LATITUDE_NAME` and `LONGITUDE_NAME` in the configuration cell.

In [ ]:
def expand_timestep_selection(selection, size):
    """Expand integers and end-exclusive 'start-stop' ranges."""
    if selection is None:
        return None
    items = selection if isinstance(selection, (list, tuple)) else [selection]
    indices = []
    for item in items:
        if isinstance(item, (int, np.integer)) and not isinstance(item, bool):
            indices.append(int(item))
        elif isinstance(item, str) and (match := re.fullmatch(r'\s*(\d+)\s*-\s*(\d+)\s*', item)):
            start, stop = map(int, match.groups())
            if stop <= start:
                raise ValueError(f'Invalid timestep range {item!r}: stop must exceed start.')
            indices.extend(range(start, stop))
        else:
            raise ValueError(f'Invalid timestep entry {item!r}.')
    if not indices or min(indices) < 0 or max(indices) >= size:
        raise IndexError(f'Timestep indices must fall in [0, {size - 1}].')
    if len(indices) != len(set(indices)):
        raise ValueError('Timestep selections contain duplicate or overlapping indices.')
    return indices


def normalize_level_indices(value, variable):
    """Return a level selection as a list while retaining negative indices."""
    if value is None:
        return None
    values = [value] if isinstance(value, (int, np.integer)) else list(value)
    if not values or any(not isinstance(index, (int, np.integer)) for index in values):
        raise ValueError(f'level_indices for {variable!r} must contain integers.')
    if len(values) != len(set(map(int, values))):
        raise ValueError(f'level_indices for {variable!r} contains duplicates.')
    return list(map(int, values))


def find_axis_name(dataset, kind, override=None):
    """Find an axis using explicit names, CF attributes, units, and aliases."""
    if override is not None:
        if override not in dataset.variables and override not in dataset.dims:
            raise KeyError(f'{kind} override {override!r} is not present in the dataset.')
        return override
    aliases = {
        'latitude': {'lat', 'latitude', 'clat', 'nav_lat', 'grid_latitude'},
        'longitude': {'lon', 'longitude', 'clon', 'nav_lon', 'grid_longitude'},
        'time': {'time', 'times', 'date', 'datetime'},
    }[kind]
    standard_name = {'latitude': 'latitude', 'longitude': 'longitude', 'time': 'time'}[kind]
    axis = {'latitude': 'Y', 'longitude': 'X', 'time': 'T'}[kind]
    scored = []
    for name, value in dataset.variables.items():
        attrs = {str(key).lower(): str(val).lower() for key, val in value.attrs.items()}
        score = 0
        score += 100 * (attrs.get('standard_name') == standard_name)
        score += 80 * (attrs.get('axis', '').upper() == axis)
        score += 50 * (str(name).lower() in aliases)
        units = attrs.get('units', '')
        if kind == 'latitude':
            score += 40 * ('degrees_north' in units)
        elif kind == 'longitude':
            score += 40 * ('degrees_east' in units)
        if score:
            scored.append((score, str(name)))
    if not scored:
        if kind == 'time':
            return None
        raise ValueError(f'Could not identify a {kind} coordinate; set its override explicitly.')
    return max(scored)[1]


def angles_in_degrees(coordinate):
    """Convert radian coordinates (common in ICON files) to degrees."""
    units = str(coordinate.attrs.get('units', '')).lower()
    values = np.asarray(coordinate.values, dtype=np.float64)
    if 'rad' in units or (np.nanmax(np.abs(values)) <= 2 * np.pi + 1e-6 and 'degree' not in units):
        values = np.rad2deg(values)
    return values


def variable_specifications(selection):
    """Flatten optional 2D/3D groups; an empty mapping means all variables."""
    entries = {str(key): value for key, value in selection.items() if str(key) != 'timesteps'}
    if any(group in entries for group in ('2D', '3D')):
        unexpected = set(entries) - {'2D', '3D'}
        if unexpected:
            raise ValueError(f'Unexpected entries beside 2D/3D groups: {sorted(unexpected)}')
        flattened = {}
        for group in ('2D', '3D'):
            for variable, specification in (entries.get(group) or {}).items():
                if variable in flattened:
                    raise ValueError(f'Variable {variable!r} occurs in more than one group.')
                flattened[str(variable)] = specification or {}
        return flattened
    return {name: specification or {} for name, specification in entries.items()}


def identify_level_dimension(data_array, spatial_dims, time_dim, specification):
    """Resolve the vertical dimension only when a level subset is requested."""
    override = specification.get('level_dimension') or LEVEL_DIMENSIONS.get(data_array.name)
    candidates = [dim for dim in data_array.dims if dim not in spatial_dims and dim != time_dim]
    if override is not None:
        if override not in candidates:
            raise ValueError(f'Level dimension {override!r} is not a non-spatial dimension of {data_array.name!r}.')
        return override
    preferred = []
    aliases = {'lev', 'level', 'levels', 'plev', 'height', 'depth', 'altitude', 'model_level'}
    for dim in candidates:
        coord = data_array.coords.get(dim)
        attrs = {} if coord is None else {str(k).lower(): str(v).lower() for k, v in coord.attrs.items()}
        if dim.lower() in aliases or attrs.get('axis', '').upper() == 'Z' or attrs.get('positive') in {'up', 'down'}:
            preferred.append(dim)
    if len(preferred) == 1:
        return preferred[0]
    if len(candidates) == 1:
        return candidates[0]
    raise ValueError(
        f'Cannot uniquely identify the level dimension of {data_array.name!r}; '
        'set LEVEL_DIMENSIONS for this variable.'
    )

## 4. HEALPix grid and remapping engine

The target contains `12 × 4**HEALPIX_LEVEL` equal-area cells. Rectilinear fields are interpolated by `xarray` after sorting latitude and adding periodic longitude edge columns. For a curvilinear or unstructured grid, the code triangulates longitude–latitude points with copies near the dateline, applies piecewise-linear interpolation to each independent time/level slice, and fills only unresolved targets from the nearest source point on the sphere.

The fallback assumes source longitude and latitude identify cell centers. It is interpolation rather than conservative remapping: use a dedicated conservative regridder when exact integral preservation is required.

In [ ]:
def build_healpix_grid(level, nested=True):
    if not isinstance(level, int) or isinstance(level, bool) or level < 0:
        raise ValueError('HEALPIX_LEVEL must be a non-negative integer.')
    nside = 2 ** level
    cell = np.arange(hp.nside2npix(nside), dtype=np.int64)
    lon, lat = hp.pix2ang(nside, cell, nest=nested, lonlat=True)
    return xr.Dataset(
        coords={
            'cell': ('cell', cell, {'long_name': 'HEALPix pixel index'}),
            'lon': ('cell', lon.astype(np.float64), {'standard_name': 'longitude', 'units': 'degrees_east'}),
            'lat': ('cell', lat.astype(np.float64), {'standard_name': 'latitude', 'units': 'degrees_north'}),
        },
        attrs={'healpix_level': level, 'healpix_nside': nside, 'healpix_ordering': 'nested' if nested else 'ring'},
    )


def remap_rectilinear(data_array, latitude, longitude, target_grid):
    """Use xarray's paired, vectorized linear interpolation."""
    lat_name, lon_name = latitude.name, longitude.name
    work = data_array.assign_coords({
        lat_name: (latitude.dims, angles_in_degrees(latitude), latitude.attrs),
        lon_name: (longitude.dims, np.mod(angles_in_degrees(longitude), 360.0), longitude.attrs),
    }).sortby(lat_name).sortby(lon_name)

    # Remove a duplicate 0/360 column, then provide periodic neighbors at both edges.
    _, unique_indices = np.unique(np.asarray(work[lon_name]), return_index=True)
    work = work.isel({lon_name: np.sort(unique_indices)})
    west = work.isel({lon_name: -1}).assign_coords({lon_name: float(work[lon_name][-1]) - 360.0})
    east = work.isel({lon_name: 0}).assign_coords({lon_name: float(work[lon_name][0]) + 360.0})
    periodic = xr.concat([west, work, east], dim=lon_name)

    target_lat = xr.DataArray(target_grid.lat.values, dims='cell')
    target_lon = xr.DataArray(np.mod(target_grid.lon.values, 360.0), dims='cell')
    indexers = {lat_name: target_lat, lon_name: target_lon}
    linear = periodic.interp(indexers, method='linear')
    nearest = periodic.interp(indexers, method='nearest', kwargs={'fill_value': None})
    result = linear.fillna(nearest)
    return result.drop_vars([name for name in (lat_name, lon_name) if name in result.coords])


def spherical_xyz(lon_degrees, lat_degrees):
    lon = np.deg2rad(lon_degrees)
    lat = np.deg2rad(lat_degrees)
    return np.column_stack((np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)))


def remap_scattered(data_array, latitude, longitude, target_grid):
    """Piecewise-linear interpolation for shared curvilinear/unstructured coordinates."""
    lat_values, lon_values = xr.broadcast(
        xr.DataArray(angles_in_degrees(latitude), dims=latitude.dims),
        xr.DataArray(angles_in_degrees(longitude), dims=longitude.dims),
    )
    spatial_dims = tuple(dict.fromkeys((*latitude.dims, *longitude.dims)))
    slice_dims = tuple(dim for dim in data_array.dims if dim not in spatial_dims)
    work = data_array.transpose(*slice_dims, *spatial_dims)
    values = np.asarray(work.values).reshape((-1, lat_values.size))

    source_lat = np.asarray(lat_values).reshape(-1)
    source_lon = np.mod(np.asarray(lon_values).reshape(-1), 360.0)
    geometry_valid = np.isfinite(source_lat) & np.isfinite(source_lon)
    source_lat, source_lon = source_lat[geometry_valid], source_lon[geometry_valid]
    values = values[:, geometry_valid]

    # Copy only a narrow strip near the dateline instead of tripling a large mesh.
    angular_spacing = np.rad2deg(np.sqrt(4 * np.pi / max(source_lon.size, 1)))
    seam_width = min(30.0, max(5.0, 4.0 * angular_spacing))
    left = np.flatnonzero(source_lon < seam_width)
    right = np.flatnonzero(source_lon > 360.0 - seam_width)
    source_indices = np.concatenate((np.arange(source_lon.size), left, right))
    extended_lon = np.concatenate((source_lon, source_lon[left] + 360.0, source_lon[right] - 360.0))
    extended_lat = np.concatenate((source_lat, source_lat[left], source_lat[right]))
    points = np.column_stack((extended_lon, extended_lat))
    triangulation = Delaunay(points, qhull_options='QJ')
    targets = np.column_stack((np.mod(target_grid.lon.values, 360.0), target_grid.lat.values))

    remapped = np.empty((values.shape[0], target_grid.sizes['cell']), dtype=np.float64)
    for field_index in tqdm(range(values.shape[0]), desc=f'{data_array.name} slices', leave=False):
        source_values = values[field_index]
        extended_values = source_values[source_indices]
        target_values = np.asarray(LinearNDInterpolator(triangulation, extended_values)(targets))
        missing = ~np.isfinite(target_values)
        finite_source = np.isfinite(source_values)
        if missing.any() and finite_source.any():
            tree = cKDTree(spherical_xyz(source_lon[finite_source], source_lat[finite_source]))
            _, nearest = tree.query(spherical_xyz(target_grid.lon.values[missing], target_grid.lat.values[missing]))
            target_values[missing] = source_values[finite_source][nearest]
        remapped[field_index] = target_values

    output_shape = tuple(work.sizes[dim] for dim in slice_dims) + (target_grid.sizes['cell'],)
    coords = {dim: work.coords[dim] for dim in slice_dims if dim in work.coords}
    return xr.DataArray(
        remapped.reshape(output_shape), dims=(*slice_dims, 'cell'), coords=coords, name=data_array.name
    )


def remap_data_array(data_array, latitude, longitude, target_grid, output_dtype):
    regular = latitude.ndim == longitude.ndim == 1 and latitude.dims != longitude.dims
    if regular:
        result = remap_rectilinear(data_array, latitude, longitude, target_grid)
    else:
        if latitude.dims != longitude.dims or latitude.shape != longitude.shape:
            raise ValueError('Curvilinear/unstructured latitude and longitude must share dimensions and shape.')
        result = remap_scattered(data_array, latitude, longitude, target_grid)
    result = result.astype(output_dtype)
    result.attrs = dict(data_array.attrs)
    result.attrs['grid_mapping'] = 'healpix'
    return result


def remap_dataset(dataset, specifications, latitude_name, longitude_name, time_dim, target_grid):
    latitude, longitude = dataset[latitude_name], dataset[longitude_name]
    spatial_dims = tuple(dict.fromkeys((*latitude.dims, *longitude.dims)))
    regular = latitude.ndim == longitude.ndim == 1 and latitude.dims != longitude.dims
    required_dims = set(spatial_dims)

    if not specifications:
        names = [
            name for name, field in dataset.data_vars.items()
            if required_dims.issubset(field.dims) and np.issubdtype(field.dtype, np.number)
        ]
        specifications = {name: {} for name in names}
    if not specifications:
        raise ValueError('No numeric variables use the detected horizontal grid.')

    timestep_indices = expand_timestep_selection(
        VARIABLES.get('timesteps'), dataset.sizes[time_dim]
    ) if time_dim is not None and VARIABLES.get('timesteps') is not None else None
    if VARIABLES.get('timesteps') is not None and time_dim is None:
        raise ValueError('Timesteps were requested, but no time dimension was found.')

    outputs = {}
    selections = []
    for name in tqdm(list(specifications), desc='Remapping variables'):
        if name not in dataset.data_vars:
            raise KeyError(f'Variable {name!r} is not a data variable in the input file.')
        field = dataset[name]
        if not required_dims.issubset(field.dims):
            raise ValueError(f'Variable {name!r} does not use horizontal dimensions {spatial_dims}.')
        if not np.issubdtype(field.dtype, np.number):
            raise TypeError(f'Variable {name!r} is not numeric and cannot be linearly remapped.')
        if timestep_indices is not None and time_dim in field.dims:
            field = field.isel({time_dim: timestep_indices})
        level_indices = normalize_level_indices(specifications[name].get('level_indices'), name)
        level_dim = None
        if level_indices is not None:
            level_dim = identify_level_dimension(field, spatial_dims, time_dim, specifications[name])
            size = field.sizes[level_dim]
            if any(index < -size or index >= size for index in level_indices):
                raise IndexError(f'level_indices for {name!r} fall outside dimension {level_dim!r} of size {size}.')
            field = field.isel({level_dim: level_indices})
        # Materialize each variable once so previewing and Zarr writing do not rerun interpolation.
        outputs[name] = remap_data_array(
            field, latitude, longitude, target_grid, OUTPUT_DTYPE
        ).load()
        selections.append({
            'variable': name, 'input dimensions': str(tuple(field.dims)),
            'selected shape': str(tuple(field.shape)), 'level dimension': level_dim or 'all/none',
            'method': 'xarray linear' if regular else 'triangulated linear + nearest fill',
        })

    output = xr.Dataset(outputs, coords={
        'cell': target_grid.cell, 'lon': target_grid.lon, 'lat': target_grid.lat,
    })
    output.attrs = dict(dataset.attrs)
    previous_history = str(output.attrs.get('history', '')).strip()
    remap_history = (
        f'{datetime.now(timezone.utc).isoformat()}: remapped from {INPUT_PATH} to '
        f'HEALPix level {HEALPIX_LEVEL} ({target_grid.attrs["healpix_ordering"]} ordering).'
    )
    output.attrs.update(target_grid.attrs)
    output.attrs['source_file'] = str(INPUT_PATH)
    output.attrs['history'] = f'{previous_history}\n{remap_history}'.strip()
    output['healpix'] = xr.DataArray(0, attrs={
        'grid_mapping_name': 'healpix',
        'healpix_level': HEALPIX_LEVEL,
        'nside': 2 ** HEALPIX_LEVEL,
        'ordering': target_grid.attrs['healpix_ordering'],
    })
    return output, pd.DataFrame(selections)

## 5. Inspect and validate the source dataset

The file remains lazily opened. This cell detects the horizontal and time coordinates, determines whether the grid is rectilinear or shared-coordinate unstructured/curvilinear, and shows the variables eligible for automatic selection. Explicitly requested variables are checked during remapping. A file without recognizable longitude and latitude coordinates requires the overrides in Section 2.

In [ ]:
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f'Input NetCDF file does not exist: {INPUT_PATH}')
if OUTPUT_DTYPE not in {'float32', 'float64'}:
    raise ValueError("OUTPUT_DTYPE must be 'float32' or 'float64'.")

source_dataset = xr.open_dataset(INPUT_PATH, decode_times=True)
latitude_name = find_axis_name(source_dataset, 'latitude', LATITUDE_NAME)
longitude_name = find_axis_name(source_dataset, 'longitude', LONGITUDE_NAME)
time_dimension = find_axis_name(source_dataset, 'time', TIME_DIMENSION)
if latitude_name in source_dataset.data_vars or longitude_name in source_dataset.data_vars:
    source_dataset = source_dataset.set_coords([latitude_name, longitude_name])

latitude, longitude = source_dataset[latitude_name], source_dataset[longitude_name]
spatial_dims = tuple(dict.fromkeys((*latitude.dims, *longitude.dims)))
eligible_variables = [
    name for name, field in source_dataset.data_vars.items()
    if set(spatial_dims).issubset(field.dims) and np.issubdtype(field.dtype, np.number)
]
grid_kind = (
    'rectilinear latitude-longitude'
    if latitude.ndim == longitude.ndim == 1 and latitude.dims != longitude.dims
    else 'curvilinear/unstructured shared coordinates'
)
display(pd.Series({
    'Input': str(INPUT_PATH),
    'Dataset dimensions': dict(source_dataset.sizes),
    'Detected grid': grid_kind,
    'Latitude / longitude': f'{latitude_name} / {longitude_name}',
    'Horizontal dimensions': spatial_dims,
    'Time dimension': time_dimension or 'none',
    'Eligible variables': ', '.join(eligible_variables),
}).to_frame('value'))

## 6. Remap the selected fields

This creates the HEALPix cell centers and applies the appropriate interpolation path. The table records the exact input shape, level handling, and interpolation method for every selected variable. Large targets grow as `12 × 4**level`; check the reported cell count before choosing a very high level.

In [ ]:
target_grid = build_healpix_grid(HEALPIX_LEVEL, HEALPIX_NESTED)
specifications = variable_specifications(VARIABLES)
remapped_dataset, selection_summary = remap_dataset(
    source_dataset, specifications, latitude_name, longitude_name, time_dimension, target_grid
)
display(selection_summary)
display(pd.Series({
    'HEALPix level': HEALPIX_LEVEL,
    'NSIDE': 2 ** HEALPIX_LEVEL,
    'Cells': target_grid.sizes['cell'],
    'Ordering': target_grid.attrs['healpix_ordering'],
    'Output dimensions': dict(remapped_dataset.sizes),
}).to_frame('value'))

## 7. Visual quality check

The first selected variable is reduced to its first time/level entry and displayed directly in HEALPix ordering. This is a quick check for coordinate swaps, radian/degree mistakes, dateline seams, or unexpected missing regions; it is not a quantitative validation.

In [ ]:
plot_variable = selection_summary.iloc[0]['variable']
plot_field = remapped_dataset[plot_variable]
plot_indexers = {dim: 0 for dim in plot_field.dims if dim != 'cell'}
plot_values = np.asarray(plot_field.isel(plot_indexers).compute())
finite = plot_values[np.isfinite(plot_values)]
if not finite.size:
    raise ValueError(f'{plot_variable!r} contains no finite remapped values.')
vmin, vmax = np.quantile(finite, [0.01, 0.99])
hp.mollview(
    plot_values, nest=HEALPIX_NESTED, title=f'{plot_variable} · HEALPix level {HEALPIX_LEVEL}',
    unit=remapped_dataset[plot_variable].attrs.get('units', ''), min=float(vmin), max=float(vmax),
    cmap='viridis',
)
plt.show()

## 8. Write and verify the Zarr store

The final dataset is chunked along time/level dimensions and HEALPix cells, then written as consolidated Zarr. Existing output is rejected unless `OVERWRITE_OUTPUT=True`. The store is reopened immediately and checked for matching variables, dimensions, HEALPix metadata, and finite data. A write failure usually indicates an unwritable output directory or insufficient storage.

In [ ]:
if OUTPUT_PATH.exists() and not OVERWRITE_OUTPUT:
    raise FileExistsError(
        f'Output already exists: {OUTPUT_PATH}. Set OVERWRITE_OUTPUT=True or choose another path.'
    )
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

chunk_sizes = {'cell': min(65536, remapped_dataset.sizes['cell'])}
for dimension, size in remapped_dataset.sizes.items():
    if dimension != 'cell':
        chunk_sizes[dimension] = 1 if dimension == time_dimension else min(size, 8)
encoded_dataset = remapped_dataset.chunk(chunk_sizes)
encoded_dataset.to_zarr(OUTPUT_PATH, mode='w', consolidated=True, zarr_format=2)

with xr.open_zarr(OUTPUT_PATH, consolidated=True) as check:
    expected_variables = set(remapped_dataset.data_vars)
    if set(check.data_vars) != expected_variables or dict(check.sizes) != dict(remapped_dataset.sizes):
        raise RuntimeError('The reopened Zarr structure does not match the remapped dataset.')
    first_name = selection_summary.iloc[0]['variable']
    first_indexers = {dim: 0 for dim in check[first_name].dims if dim != 'cell'}
    finite_count = int(np.isfinite(check[first_name].isel(first_indexers).values).sum())
    if finite_count == 0:
        raise RuntimeError('The reopened Zarr store contains no finite values in its first field.')
    written_summary = pd.Series({
        'Zarr store': str(OUTPUT_PATH.resolve()),
        'Variables': ', '.join(name for name in check.data_vars if name != 'healpix'),
        'Dimensions': dict(check.sizes),
        'HEALPix level': check.attrs.get('healpix_level'),
        'Finite values in checked map': finite_count,
    })
display(written_summary.to_frame('value'))
source_dataset.close()

## Notes and extensions

The output is ready to use as the highest-resolution input store for the compression training and inference notebooks. Those notebooks derive their lower HEALPix inputs dynamically. This notebook samples values at HEALPix cell centers and therefore does not conserve area-integrated quantities such as accumulated precipitation exactly. For conservation-critical workflows, replace the interpolation kernel with a conservative remapper while retaining the selection, metadata, and Zarr-writing sections.